In [1]:
# >>>>>>>>>> Init <<<<<<<<<<
from PIL import Image, ImageOps, ImageEnhance
import matplotlib.pyplot as plt
import matplotlib as mpl

In [7]:
def func_show_box(i_frame, i_ball):
    im = Image.open(f"local/for_train/train/images/{i_frame}.jpg") 
    width, height = im.size
    print(f"Кадр {i_frame}:\n")
    with open(f"local/for_train/train/labels/d{i_frame}.txt", 'r') as f:
        xywh, x1, x2, y1, y2 = [], [], [], [], [] 
        for i, line in enumerate(f):
            if line != '':
                print(line.split())
                id_,x,y,w,h = line.split()
                xywh.append([int(id_), float(x), float(y), float(w), float(h)])
                x1.append(xywh[i][1] - xywh[i][3]/2)
                x2.append(xywh[i][1] + xywh[i][3]/2)
                y1.append(xywh[i][2] - xywh[i][4]/2)
                y2.append(xywh[i][2] + xywh[i][4]/2)
                print(f"Центр [{int(xywh[i][1])}, {int(xywh[i][2])}]\nДиаметр {int(xywh[i][3])}")

    with open(f"local/for_train/train/labels/{i_frame}.txt", 'w') as f:
        if len(xywh) > 0:
            for i in range(len(xywh)):
                ending = "" if i==len(xywh)-1 else "\n"
                f.write(f"{xywh[i][0]} {xywh[i][1]/width} {xywh[i][2]/height} {xywh[i][3]/width} {xywh[i][4]/height}{ending}")
        else:
            f.write(f"")

    if len(xywh) > 0:
        if ZOOM:
            print((x1[i_ball] - d, y1[i_ball] - d, x2[i_ball] + d,  y2[i_ball] + d))
            imcrop = im.crop((x1[i_ball] - d, 
                              y1[i_ball] - d, 
                              x2[i_ball] + d, 
                              y2[i_ball] + d))
            plt.imshow(imcrop)
            plt.plot([d, x2[i_ball]-x1[i_ball]+d], [d, d], c='r')
            plt.plot([d, x2[i_ball]-x1[i_ball]+d], [y2[i_ball]-y1[i_ball]+d, y2[i_ball]-y1[i_ball]+d], c='r')
            plt.plot([d, d], [d, y2[i_ball]-y1[i_ball]+d], c='r')
            plt.plot([x2[i_ball]-x1[i_ball]+d, x2[i_ball]-x1[i_ball]+d], [d, y2[i_ball]-y1[i_ball]+d], c='r')
        else:
            for i in range(len(xywh)):
                plt.plot([x1[i], x2[i]], [y1[i], y1[i]], c='r')
                plt.plot([x1[i], x2[i]], [y2[i], y2[i]], c='r')
                plt.plot([x1[i], x1[i]], [y1[i], y2[i]], c='r')
                plt.plot([x2[i], x2[i]], [y1[i], y2[i]], c='r')
    if len(xywh) == 0 or (not ZOOM):
        plt.imshow(im)
    mpl.rcParams['figure.dpi'] = 150
    # plt.show()

##### <u>Создание обучающей выборки</u> 

In [13]:
d = 50; ZOOM = False
i_frame = 630; i_ball = 0

func_show_box(i_frame=i_frame, i_ball=i_ball)

FileNotFoundError: [Errno 2] No such file or directory: '/home/orlov/Desktop/my-matlab/YoloOpticalNav26/PythonPreProcessing/local/for_train/train/images/630.jpg'

In [ ]:
# Awesome 1 ball in the video
a = {'obj': 0, 'x': 292
              ,'y': 190
              ,'d': 260
              ,}
with open(f"local/for_train/train/labels/d{i_frame}.txt", 'w') as f:
    f.write(f"{a['obj']} {a['x']} {a['y']} {a['d']} {a['d']}")

In [38]:
# Awesome 2 balls in the video
a = {'obj': [0,0], 'x': [657, 962]
                  ,'y': [197, 405]
                  ,'d': [132, 344]}
with open(f"local/for_train/train/labels/d{i_frame}.txt", 'w') as f:
    f.write(f"{a['obj'][0]} {a['x'][0]} {a['y'][0]} {a['d'][0]} {a['d'][0]}\n")
    f.write(f"{a['obj'][1]} {a['x'][1]} {a['y'][1]} {a['d'][1]} {a['d'][1]}")

In [4]:
# No any balls in video
with open(f"local/for_train/train/labels/d{i_frame}.txt", 'w') as f:
    f.write(f"")

##### <u>Искусственное увеличение обучающей выборки</u>   

In [14]:
n = 151

p1 = lambda i: f"local/for_train/train/images/{i}.jpg"
p2 = [lambda i: f"local/for_train/train/labels/{i}.txt",
      lambda i: f"local/for_train/train/labels/d{i}.txt"]

j = n
for i in range(1, n+1):
    im0 = Image.open(p1(i))
    a = [[1, 1], [im0.size[0], im0.size[1]]]
    x_pix, x_rel = [], []
    with open(p2[0](i),"r") as f:
        for line in f:
            tmp = line.split()
            x_rel.append([int(tmp[0])]+[float(tmp[i]) for i in range(1,5)])
    with open(p2[1](i),"r") as f:
        for line in f:
            tmp = line.split()
            x_pix.append([int(tmp[i]) for i in range(0,5)])
    x = [x_rel, x_pix]
    # j = 0
    
    for im in [im0, 
               ImageEnhance.Sharpness(im0).enhance(4),
               ImageEnhance.Contrast(im0).enhance(2)]:
        # Flip
        j += 1
        ImageOps.flip(im).save(p1(j))
        for k in range(2):  # in relative [0,1], in pixels [0,1920]
            with open(p2[k](j), 'w') as f:
                if len(x[k]) > 0:
                    for il, l in enumerate(x[k]):  # lines in label.txt
                        ending = "" if il==len(x[k])-1 else "\n"
                        f.write(f"{l[0]} {l[1]} {a[k][1]-l[2]} {l[3]} {l[4]}{ending}")
                else:
                    f.write("")

        # Mirror
        j += 1
        ImageOps.mirror(im).save(p1(j))
        for k in range(2):  # in relative [0,1], in pixels [0,1920]
            with open(p2[k](j), 'w') as f:
                if len(x[k]) > 0:
                    for il, l in enumerate(x[k]):  # lines in label.txt
                        ending = "" if il==len(x[k])-1 else "\n"
                        f.write(f"{l[0]} {a[k][0]-l[1]} {l[2]} {l[3]} {l[4]}{ending}")
                else:
                    f.write("")

        # Flip + mirror
        j += 1
        ImageOps.mirror(ImageOps.flip(im)).save(p1(j))
        for k in range(2):  # in relative [0,1], in pixels [0,1920]
            with open(p2[k](j), 'w') as f:
                if len(x[k]) > 0:
                    for il, l in enumerate(x[k]):  # lines in label.txt
                        ending = "" if il==len(x[k])-1 else "\n"
                        f.write(f"{l[0]} {a[k][0]-l[1]} {a[k][1]-l[2]} {l[3]} {l[4]}{ending}")
                else:
                    f.write("")

##### <u>Обучение YOLO-11</u>   

In [15]:
from ultralytics import YOLO
import cv2
import numpy as np

# Загрузка модели YOLOv8

# model = YOLO("yolo11n.pt")
model = YOLO('local/yolo_balls_11_old.pt')

# Список цветов для различных классов
colors = [(255, 0, 0), (0, 255, 0)]

In [16]:
# nvidia-smi - Проверка видимости видеокарты драйверами
# 23.01.2026 - ??? epoch
train_res = model.train(data="local/for_train/fortrain.yaml", epochs=500)

New https://pypi.org/project/ultralytics/8.4.7 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.158 🚀 Python-3.8.10 torch-2.4.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3070 Ti, 7848MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=local/for_train/fortrain.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=local/yolo_balls_11_old.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train6, nbs=64, nms

train: Scanning /home/orlov/Desktop/my-matlab/YoloOpticalNav26/PythonPreProcessing/local/for_train/train/labels... 1359 images, 171 backgrounds, 0 corrupt: 10

train: New cache created: /home/orlov/Desktop/my-matlab/YoloOpticalNav26/PythonPreProcessing/local/for_train/train/labels.cache


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4303.8±2303.5 MB/s, size: 403.0 KB)


val: Scanning /home/orlov/Desktop/my-matlab/YoloOpticalNav26/PythonPreProcessing/local/for_train/val/labels... 151 images, 19 backgrounds, 0 corrupt: 100%|███

...


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


    447/500      2.18G     0.2884     0.1753     0.7738         27        640: 100%|██████████| 85/85 [00:07<00:00, 10.71it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:00<00:00, 13.59it/s]

                   all        151        169          1      0.999      0.995      0.965



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


    448/500      2.18G     0.2796     0.1727     0.7746         35        640:  80%|████████  | 68/85 [00:06<00:01, 11.45it/s]

In [17]:
# СОХРАНЕНИЕ
model.save('local/yolo_balls_11_new.pt')